In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/processed/air_quality_cleaned.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (2763, 22)


In [2]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values: 0
Duplicate rows: 0


In [3]:
expected_columns = [
    "city",
    "date",
    "aqi",
    "pm25",
    "pm10",
    "no2",
    "so2",
    "co",
    "o3",
    "year",
    "month",
    "day",
    "day_of_week",
    "day_of_year",
    "is_weekend",
    "aqi_lag_1",
    "aqi_lag_3",
    "aqi_lag_7",
    "aqi_rolling_mean_3",
    "aqi_rolling_mean_7",
    "aqi_rolling_std_7",
    "target_aqi"
]

missing_columns = [
    column for column in expected_columns
    if column not in df.columns
]

print("Missing required columns:", missing_columns)

Missing required columns: []


In [4]:
print("Negative AQI:", (df["aqi"] < 0).sum())
print("AQI above 500:", (df["aqi"] > 500).sum())

print("Negative PM2.5:", (df["pm25"] < 0).sum())
print("Negative PM10:", (df["pm10"] < 0).sum())
print("Negative NO2:", (df["no2"] < 0).sum())
print("Negative SO2:", (df["so2"] < 0).sum())
print("Negative CO:", (df["co"] < 0).sum())
print("Negative O3:", (df["o3"] < 0).sum())

Negative AQI: 0
AQI above 500: 0
Negative PM2.5: 0
Negative PM10: 0
Negative NO2: 0
Negative SO2: 0
Negative CO: 0
Negative O3: 0


In [5]:
import great_expectations as gx

print("Great Expectations version:", gx.__version__)

Great Expectations version: 1.8.1


In [6]:
import great_expectations as gx

print("Great Expectations version:", gx.__version__)

context = gx.get_context()
print("Great Expectations context created successfully")

Great Expectations version: 1.8.1
Great Expectations context created successfully


In [7]:
suite_name = "mumbai_aqi_data_quality_suite"

suite = gx.ExpectationSuite(name=suite_name)

suite = context.suites.add(suite)

print("Expectation Suite created:", suite.name)

Expectation Suite created: mumbai_aqi_data_quality_suite


In [8]:
required_columns = [
    "city",
    "date",
    "aqi",
    "pm25",
    "pm10",
    "no2",
    "so2",
    "co",
    "o3",
    "year",
    "month",
    "day",
    "day_of_week",
    "day_of_year",
    "is_weekend",
    "aqi_lag_1",
    "aqi_lag_3",
    "aqi_lag_7",
    "aqi_rolling_mean_3",
    "aqi_rolling_mean_7",
    "aqi_rolling_std_7",
    "target_aqi"
]

for column in required_columns:
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(
            column=column
        )
    )

print(f"Added {len(required_columns)} non-null expectations")

Added 22 non-null expectations


In [9]:
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="aqi",
        min_value=0,
        max_value=500
    )
)

print("AQI range expectation added")

AQI range expectation added


In [10]:
pollutant_columns = [
    "pm25",
    "pm10",
    "no2",
    "so2",
    "co",
    "o3"
]

for column in pollutant_columns:
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeBetween(
            column=column,
            min_value=0
        )
    )

print("Pollutant range expectations added")

Pollutant range expectations added


In [11]:
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="is_weekend",
        value_set=[0, 1]
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="month",
        min_value=1,
        max_value=12
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="day_of_week",
        min_value=0,
        max_value=6
    )
)

print("Calendar feature expectations added")

Calendar feature expectations added


In [12]:
suite.save()

print("Expectation Suite saved successfully")

Expectation Suite saved successfully


In [13]:
data_source = context.data_sources.add_pandas(
    name="mumbai_aqi_pandas"
)

print("Data source created:", data_source.name)

Data source created: mumbai_aqi_pandas


In [14]:
data_asset = data_source.add_dataframe_asset(
    name="mumbai_aqi_cleaned"
)

print("Data asset created:", data_asset.name)

Data asset created: mumbai_aqi_cleaned


In [15]:
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    "mumbai_aqi_batch"
)

print("Batch definition created:", batch_definition.name)

Batch definition created: mumbai_aqi_batch


In [16]:
validation_definition = gx.ValidationDefinition(
    data=batch_definition,
    suite=suite,
    name="mumbai_aqi_validation"
)

validation_definition = context.validation_definitions.add(
    validation_definition
)

print(
    "Validation definition created:",
    validation_definition.name
)

Validation definition created: mumbai_aqi_validation


In [17]:
checkpoint = gx.Checkpoint(
    name="mumbai_aqi_checkpoint",
    validation_definitions=[
        validation_definition
    ]
)

checkpoint = context.checkpoints.add(checkpoint)

print("Checkpoint created:", checkpoint.name)

Checkpoint created: mumbai_aqi_checkpoint


In [18]:
validation_results = checkpoint.run(
    batch_parameters={
        "dataframe": df
    }
)

print(validation_results)

Calculating Metrics: 100%|██████████| 168/168 [00:00<00:00, 4037.21it/s]

run_id={"run_name": null, "run_time": "2026-08-26T22:37:29.162626+05:30"} run_results={ValidationResultIdentifier::mumbai_aqi_data_quality_suite/__none__/20260826T170729.162626Z/mumbai_aqi_pandas-mumbai_aqi_cleaned: {
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "mumbai_aqi_pandas-mumbai_aqi_cleaned",
          "column": "city"
        },
        "meta": {},
        "id": "b70c0287-767b-4e63-a828-f8a548110ef0",
        "severity": "critical"
      },
      "result": {
        "element_count": 2763,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
  

In [19]:
print(validation_results.success)

True
